# Phase 1 — Data Acquisition

Queries Sentinel-2 L2A imagery for Delhi NCR across two temporal
windows (Jan–Feb 2024 and Jan–Feb 2026), reports honest metadata
(image count, cloud %, dates), visually sanity-checks the composites,
and exports them to Google Drive for Phase 2 preprocessing.

Run this after 00_colab_setup.ipynb, or run the setup cells below
directly — they are repeated here for convenience.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.5 MB/s eta 0:00:00


In [ ]:
import ee

ee.Authenticate()
ee.Initialize(project='stellar-stream-492412-p9')
print('Earth Engine initialized successfully.')

Earth Engine initialized successfully.


In [ ]:
import os, sys

REPO_URL = 'https://github.com/karan02566-prog/delhi-ncr-satellite-change.git'
REPO_DIR = '/content/delhi-ncr-satellite-change'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

Cloning into '/content/delhi-ncr-satellite-change'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 19 (delta 4), reused 18 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 6.75 KiB | 6.75 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/delhi-ncr-satellite-change


In [ ]:
from src.data.gee_download import (
    get_delhi_ncr_roi,
    get_sentinel2_collection,
    get_collection_metadata,
    get_median_composite,
    export_to_drive,
)

roi = get_delhi_ncr_roi()
print('ROI defined:', roi.getInfo())

ROI defined: {'type': 'Polygon', 'coordinates': [[[76.75, 28.3], [77.6, 28.3], [77.6, 28.95], [76.75, 28.95], [76.75, 28.3]]]}


## T1 (Jan–Feb 2024)

In [ ]:
t1_collection = get_sentinel2_collection('2024-01-01', '2024-02-15', roi, cloud_threshold=20)
t1_metadata = get_collection_metadata(t1_collection)
print('T1 metadata:', t1_metadata)

T1 metadata: {'image_count': 3, 'cloud_percentages': [7.890186, 9.201495, 1.670303], 'acquisition_dates': ['2024-01-29', '2024-02-08', '2024-02-08']}


## T2 (Jan–Feb 2026)

In [ ]:
t2_collection = get_sentinel2_collection('2026-01-01', '2026-02-15', roi, cloud_threshold=20)
t2_metadata = get_collection_metadata(t2_collection)
print('T2 metadata:', t2_metadata)

T2 metadata: {'image_count': 12, 'cloud_percentages': [18.525642, 7.385586, 1.120504, 13.176167, 1.65249, 17.445731, 5.768508, 12.523343, 17.519999, 2.817737, 13.296776, 0.00136], 'acquisition_dates': ['2026-01-05', '2026-01-05', '2026-01-18', '2026-01-18', '2026-01-18', '2026-02-07', '2026-02-07', '2026-02-07', '2026-02-07', '2026-02-07', '2026-02-07', '2026-02-07']}


## Build composites and visually sanity-check

In [ ]:
t1_composite = get_median_composite(t1_collection)
t2_composite = get_median_composite(t2_collection)

import geemap

Map = geemap.Map()
Map.centerObject(roi, 10)

vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map.addLayer(t1_composite, vis_params, 'T1 - Jan-Feb 2024')
Map.addLayer(t2_composite, vis_params, 'T2 - Jan-Feb 2026')

roi_outline = ee.Image().byte().paint(featureCollection=ee.FeatureCollection([ee.Feature(roi)]), color=1, width=2)
Map.addLayer(roi_outline, {'palette': 'red'}, 'ROI outline')

Map

Map(center=[28.625327425819, 77.1749999999999], controls=(WidgetControl(options=['position', 'transparent_bg']…

## Export composites to Google Drive

Exports are asynchronous. Check progress under the Tasks tab at
https://code.earthengine.google.com or via `ee.batch.Task.list()`.

In [ ]:
t1_task = export_to_drive(t1_composite, 'delhi_ncr_t1_2024', roi)
t2_task = export_to_drive(t2_composite, 'delhi_ncr_t2_2026', roi)

print('T1 export task started:', t1_task.status())
print('T2 export task started:', t2_task.status())

T1 export task started: {'state': 'READY', 'description': 'delhi_ncr_t1_2024', 'priority': 100, 'creation_timestamp_ms': 1787121777313, 'update_timestamp_ms': 1787121777313, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'VTN4B5QBGH642BARS4HZKOO4', 'name': 'projects/stellar-stream-492412-p9/operations/VTN4B5QBGH642BARS4HZKOO4'}
T2 export task started: {'state': 'READY', 'description': 'delhi_ncr_t2_2026', 'priority': 100, 'creation_timestamp_ms': 1787121777801, 'update_timestamp_ms': 1787121777801, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'I4ZWQ2PYSOE5OMPCL736UJTL', 'name': 'projects/stellar-stream-492412-p9/operations/I4ZWQ2PYSOE5OMPCL736UJTL'}


In [ ]:
print('T1:', t1_task.status()['state'])
print('T2:', t2_task.status()['state'])


T1: COMPLETED
T2: RUNNING


In [ ]:
!ls /content/drive/MyDrive/


'Colab Notebooks'


## Save metadata for the data validation report

This writes the real, retrieved metadata (not invented numbers) to
Drive so it can be referenced in the Phase 1 validation report.

In [ ]:
import json

metadata_out = {
    'roi_bbox': [76.75, 28.30, 77.60, 28.95],
    't1_window': ['2024-01-01', '2024-02-15'],
    't2_window': ['2026-01-01', '2026-02-15'],
    't1_metadata': t1_metadata,
    't2_metadata': t2_metadata,
}

os.makedirs('/content/drive/MyDrive/delhi_ncr_change_detection', exist_ok=True)
with open('/content/drive/MyDrive/delhi_ncr_change_detection/phase1_metadata.json', 'w') as f:
    json.dump(metadata_out, f, indent=2)

print('Saved phase1_metadata.json to Drive.')
print(json.dumps(metadata_out, indent=2))

Saved phase1_metadata.json to Drive.
{
  "roi_bbox": [
    76.75,
    28.3,
    77.6,
    28.95
  ],
  "t1_window": [
    "2024-01-01",
    "2024-02-15"
  ],
  "t2_window": [
    "2026-01-01",
    "2026-02-15"
  ],
  "t1_metadata": {
    "image_count": 3,
    "cloud_percentages": [
      7.890186,
      9.201495,
      1.670303
    ],
    "acquisition_dates": [
      "2024-01-29",
      "2024-02-08",
      "2024-02-08"
    ]
  },
  "t2_metadata": {
    "image_count": 12,
    "cloud_percentages": [
      18.525642,
      7.385586,
      1.120504,
      13.176167,
      1.65249,
      17.445731,
      5.768508,
      12.523343,
      17.519999,
      2.817737,
      13.296776,
      0.00136
    ],
    "acquisition_dates": [
      "2026-01-05",
      "2026-01-05",
      "2026-01-18",
      "2026-01-18",
      "2026-01-18",
      "2026-02-07",
      "2026-02-07",
      "2026-02-07",
      "2026-02-07",
      "2026-02-07",
      "2026-02-07",
      "2026-02-07"
    ]
  }
}


In [ ]:
!ls /content/drive/MyDrive/
!ls /content/drive/MyDrive/delhi_ncr_change_detection/

'Colab Notebooks'   delhi_ncr_change_detection
phase1_metadata.json


In [ ]:
import ee
print("Earth Engine account info:")
print(ee.data.getAssetRoots())

import subprocess
result = subprocess.run(['cat', '/content/drive/.tmp/config'], capture_output=True, text=True)
print("Drive mount check:", result.stdout, result.stderr)

Earth Engine account info:


EEException: Asset "projects/stellar-stream-492412-p9/assets" not found.

In [ ]:
import ee
ee.Reset()
ee.Authenticate(force=True)
ee.Initialize(project='stellar-stream-492412-p9')
print("Re-authenticated.")

Re-authenticated.


In [ ]:
t1_task = export_to_drive(t1_composite, 'delhi_ncr_t1_2024', roi)
t2_task = export_to_drive(t2_composite, 'delhi_ncr_t2_2026', roi)
print(t1_task.status())
print(t2_task.status())

{'state': 'READY', 'description': 'delhi_ncr_t1_2024', 'priority': 100, 'creation_timestamp_ms': 1787123567397, 'update_timestamp_ms': 1787123567397, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'X2KVVIY4OMXO3OFDO2KKART5', 'name': 'projects/stellar-stream-492412-p9/operations/X2KVVIY4OMXO3OFDO2KKART5'}
{'state': 'READY', 'description': 'delhi_ncr_t2_2026', 'priority': 100, 'creation_timestamp_ms': 1787123567998, 'update_timestamp_ms': 1787123567998, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'EZ527QDUCDTYOMLA2K6PIHTF', 'name': 'projects/stellar-stream-492412-p9/operations/EZ527QDUCDTYOMLA2K6PIHTF'}


In [ ]:
print('T1:', t1_task.status()['state'])
print('T2:', t2_task.status()['state'])

T1: COMPLETED
T2: COMPLETED
